# 05: Embedding Paradigms

Builds and compares the three Section 3.4 embedding paradigms (FT-Transformer, SubTab, SCARF) on top of the already-fitted encoders from `04_feature_encoding.ipynb`.

## Setup

In [ ]:
%matplotlib inline

import os

# torch bundles its own OpenMP runtime (libiomp5md.dll) while
# numpy/pandas/scikit-learn pull in MKL's (libomp.dll/libiomp5md.dll) --
# loading both into one process aborts the kernel outright ("OMP: Error
# #15: Initializing ..., but found ... already initialized"), not a
# Python exception, so it shows up as a bare kernel crash with no
# traceback. Must be set before numpy/torch are imported anywhere below.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import config

FINAL_DIR = config.FINAL_DIR

sns.set_theme(style="whitegrid")

from src.features.target_split import build_feature_target_split
from src.features.pipeline import FittedEncoders

## 1. Load active candidate

In [ ]:
CANDIDATES = {
    "br_gt0": {
        "train": FINAL_DIR / "tmdb_br_gt0_train.parquet",
        "val": FINAL_DIR / "tmdb_br_gt0_val.parquet",
        "test": FINAL_DIR / "tmdb_br_gt0_test.parquet",
    },
    "nbr_gt5": {
        "train": FINAL_DIR / "tmdb_nbr_gt5_train.parquet",
        "val": FINAL_DIR / "tmdb_nbr_gt5_val.parquet",
        "test": FINAL_DIR / "tmdb_nbr_gt5_test.parquet",
    },
}

In [ ]:
# Single switch: "br_gt0" is the primary candidate, "nbr_gt5" the
# fallback. Every later cell reads train_df/val_df/test_df (or
# ACTIVE_CANDIDATE for naming saved artifacts) instead of a literal
# candidate name, so switching candidates means editing only this line.
ACTIVE_CANDIDATE = "br_gt0"
# ACTIVE_CANDIDATE = "nbr_gt5"

In [ ]:
train_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["train"])
val_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["val"])
test_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["test"])

FINAL_SPLITS = {"train": train_df, "val": val_df, "test": test_df}

train_split = build_feature_target_split(train_df)

ARTIFACTS_DIR = REPO_ROOT / "data" / "artifacts" / ACTIVE_CANDIDATE
fitted_encoders = FittedEncoders.fit(FINAL_SPLITS, train_split, ARTIFACTS_DIR)

print(f"Active candidate: {ACTIVE_CANDIDATE}")
print(f"  train: {train_df.shape}")
print(f"  val:   {val_df.shape}")
print(f"  test:  {test_df.shape}")
print(f"fitted_encoders.dims: {fitted_encoders.dims}")

## 2. Paradigm hyperparameter configs

Instantiate all four `src/models/paradigm_config.py` dataclasses and print every field before anything trains on them -- a quick eyeball check against the papers they're cited from.

In [ ]:
import dataclasses

from src.models.paradigm_config import (
    FTTransformerConfig,
    SubTabConfig,
    SCARFConfig,
    SharedTrainingConfig,
)

ft_transformer_config = FTTransformerConfig()
subtab_config = SubTabConfig()
scarf_config = SCARFConfig()
shared_training_config = SharedTrainingConfig()

for name, cfg in [
    ("FTTransformerConfig", ft_transformer_config),
    ("SubTabConfig", subtab_config),
    ("SCARFConfig", scarf_config),
    ("SharedTrainingConfig", shared_training_config),
]:
    print(f"--- {name} ---")
    for key, value in dataclasses.asdict(cfg).items():
        print(f"  {key}: {value!r}")
    print()

## 3. FT-Transformer

How do the frozen SBERT vectors enter the token sequence?

`overview`/`original_title` each arrive from `fitted_encoders.transform()` as a frozen 768-d SBERT vector (`fitted_encoders.dims["overview"] == fitted_encoders.dims["original_title"] == 768`). Every other token in FT-Transformer's sequence -- numeric (PLE-then-linear), categorical (embedding lookup), list-pooled -- is built at `d_token`. Two options for how the text tokens join that sequence:

**(a) Used as-is, as two large 768-d tokens.**
- Pros: no information loss from the frozen SBERT vector -- the full representation reaches self-attention; no extra trainable compression step between the frozen encoder and the model.
- Cons: self-attention requires every token in the sequence to share the same width, so this is only actually implementable by setting `d_token=768` *globally* -- not just for the two text tokens, but for numeric, categorical, and list tokens too. That silently abandons Gorishniy et al.'s own `d_token=192` defaulf. It also inflates every attention block's parameter count and compute cost by roughly 768/192 = 4x per token width dimension, for a width inherited from SBERT's arbitrary output size, not chosen for FT-Transformer.

**(b) Projected down to `d_token` (192) via a small trainable `nn.Linear(768, d_token)` per text field.**
- Pros: keeps `d_token=192`, respecting Table 12's default; every token in the sequence stays at one uniform width, matching FT-Transformer's architecture as published rather than special-casing it; the projection is itself trainable, so the model learns which directions in the frozen 768-d SBERT space matter for this task instead of passing the whole undifferentiated vector through; parameter/compute cost stays at the paper's intended scale.
- Cons: introduces a small lossy compression (768 -> 192) before self-attention ever sees the text tokens; adds new trainable parameters (768x192 + bias, per text field) that have to be learned from this dataset, unlike SBERT's own frozen weights; whether the two text fields share one projection or each gets its own is a further decision left for the actual model code, not resolved here.

**Decision: option (b).** Self-attention's shared-width requirement means option (a) was never really a free choice -- it was a choice to override `d_token` globally. `nn.Linear(768, d_token)` projections for `overview`/`original_title` proceed as the design; no model code is written yet, this cell records the decision and reasoning only.

### Sanity check: n_tokens

`fitted_encoders.dims` has one entry per `transform()` key (9 for `br_gt0`: `numeric`, `original_language`, `overview`, `original_title`, and 5 list fields) -- but `n_tokens` is NOT `len(dims)`, because the single `"numeric"` key represents one *combined* width that `FeatureTokenizer` re-slices back into one token per `numeric_cols` entry plus one token per `bypass_cols` entry. Every other `dims` key maps to exactly one token. So:

`n_tokens = len(numeric_cols) + len(bypass_cols) + 1 (language) + len(list_cols) + 2 (text)`

equivalently `(len(dims) - 1) + len(numeric_cols) + len(bypass_cols)` -- computed both ways below and checked against `FeatureTokenizer`'s actual output.

In [ ]:
from src.models.ft_transformer import FeatureTokenizer

tokenizer = FeatureTokenizer(fitted_encoders, d_token=ft_transformer_config.d_token)
batch = fitted_encoders.transform(train_df)
tokens = tokenizer(batch)

column_groups = fitted_encoders.column_groups
n_tokens_by_groups = (
    len(column_groups.numeric_cols)
    + len(column_groups.bypass_cols)
    + 1  # original_language
    + len(column_groups.list_cols)
    + 2  # overview, original_title
)
n_tokens_by_dims = (
    (len(fitted_encoders.dims) - 1)  # every dims key except "numeric" is one token
    + len(column_groups.numeric_cols)
    + len(column_groups.bypass_cols)
)

print(f"dims keys ({len(fitted_encoders.dims)}): {list(fitted_encoders.dims.keys())}")
print(f"n_tokens via column_groups: {n_tokens_by_groups}")
print(f"n_tokens via dims key count: {n_tokens_by_dims}")
print(f"FeatureTokenizer actual output: {tokens.shape[1]}")

assert n_tokens_by_groups == n_tokens_by_dims == tokens.shape[1], (
    f"n_tokens mismatch: by_groups={n_tokens_by_groups}, "
    f"by_dims={n_tokens_by_dims}, actual={tokens.shape[1]}"
)
print("PASSED: all three counts agree")

Beyond `n_tokens` alone, `FeatureTokenizer.forward()`'s full output must be exactly `(batch_size, n_tokens, d_token)` -- one token per feature, each `d_token` wide, for every row in the batch (Prompt 2.2's check). Reusing `tokenizer`/`batch`/`tokens` from the cell above, `batch_size` here is `len(train_df)` since `batch = fitted_encoders.transform(train_df)` ran on the whole training set as one call, not a mini-batch.

In [ ]:
n_rows = len(train_df)
expected_shape = (n_rows, n_tokens_by_groups, ft_transformer_config.d_token)

print(f"batch size (rows in train_df): {n_rows}")
print(f"FeatureTokenizer output shape: {tuple(tokens.shape)}")
print(f"expected shape: {expected_shape}")

assert tokens.shape == expected_shape, (
    f"shape mismatch: got {tuple(tokens.shape)}, expected {expected_shape}"
)
print("PASSED: output shape is (batch_size, n_tokens, d_token) as predicted")

### Sanity check: parameter count

Instantiate `FTTransformerModel` from `fitted_encoders.dims` (drives `FeatureTokenizer`'s per-field embedding/projection tables) and `ft_transformer_config`, then count total parameters. Gorishniy et al. (2021, Table 12) report 929K parameters for FT-Transformer -- but that's on their own benchmark datasets, with their own feature counts and vocabulary sizes, not this one. This dataset has more embedding-heavy fields (5 list fields plus `original_language`, each its own vocabulary -sized `nn.Embedding` table) and two 768->192 text projections, so matching 929K isn't the bar -- staying in the same order of magnitude is.

In [ ]:
from src.models.ft_transformer import FTTransformerModel

model = FTTransformerModel(fitted_encoders, ft_transformer_config)
total_params = sum(p.numel() for p in model.parameters())

print(f"total parameters: {total_params:,}")
print(f"Gorishniy et al. (2021), Table 12: 929,000")
print(f"ratio: {total_params / 929_000:.2f}x")

### Train FT-Transformer

In [ ]:
from src.models.train_ft_transformer import train_ft_transformer

model, train_losses, val_losses = train_ft_transformer(
    model, train_df, val_df, fitted_encoders, ft_transformer_config, shared_training_config
)

print(
    f"Epochs run: {len(train_losses)} "
    f"(max_epochs={shared_training_config.max_epochs}, patience={shared_training_config.patience})"
)
print(f"Final train MSE: {train_losses[-1]:.4f}")
print(f"Val MSE (restored/best epoch): {min(val_losses):.4f}")

**Ad hoc sanity check only -- not the final asset.** Prompt 5.5's shared diagnostics logging will later replace this plot with the identical-across-paradigms version (same axes/styling for FT-Transformer, SubTab, and SCARF alike). This cell just gives a quick look at whether `train_losses`/`val_losses` moved the way early stopping's logic assumes, before that shared logger exists.

In [ ]:
fig, ax = plt.subplots()
ax.plot(train_losses, label="train MSE")
ax.plot(val_losses, label="val MSE")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE (vote_average)")
ax.set_title("FT-Transformer training curve (ad hoc -- see note above)")
ax.legend()
plt.show()

### Save FT-Transformer embeddings

`model.embed()` (Eq. 3.9) run over `train_df`/`val_df`/`test_df` in batches, under `torch.no_grad()` -- inference only, no gradient tracking needed once the model is trained. Each split's `(N, d_token)` embedding matrix is saved to `ARTIFACTS_DIR / "models" / "ft_transformer" / f"{split_name}.parquet"`, mirroring `04_feature_encoding.ipynb`'s `RAW_MODEL_DIR`/`CLASSICAL_MODEL_DIR` pattern: one row per input row, in the same order as the split's DataFrame -- no `id`/target column saved alongside, same as the raw/classical baselines, so downstream code aligns by row position against the original split parquet files. Columns are named `dim_0..dim_{d_token-1}`; there's no natural per-dimension feature name the way raw/classical's one-hot/multi-hot columns have one.

In [ ]:
import torch

FT_TRANSFORMER_MODEL_DIR = ARTIFACTS_DIR / "models" / "ft_transformer"
FT_TRANSFORMER_MODEL_DIR.mkdir(parents=True, exist_ok=True)

model.eval()
for split_name, df in FINAL_SPLITS.items():
    batch = fitted_encoders.transform(df)
    embedding_chunks = []
    with torch.no_grad():
        for start in range(0, len(df), ft_transformer_config.batch_size):
            end = min(start + ft_transformer_config.batch_size, len(df))
            chunk = {key: value[start:end] for key, value in batch.items()}
            embedding_chunks.append(model.embed(chunk).numpy())
    embeddings = np.concatenate(embedding_chunks, axis=0)

    embeddings_df = pd.DataFrame(
        embeddings, columns=[f"dim_{i}" for i in range(embeddings.shape[1])]
    )
    path = FT_TRANSFORMER_MODEL_DIR / f"{split_name}.parquet"
    embeddings_df.to_parquet(path, index=False)
    print(f"Saved: {path} ({embeddings_df.shape})")

## 4. SubTab

### SVD diagnostic: does 80% variance fit in 300 components?

Disposable check, nothing saved. `SubTabConfig.svd_variance_threshold` (80%) and `svd_n_components_ceiling` (300) are Prompt 3.1/3.2's own numbers, not paper-sourced (see `SubTabConfig`'s docstring) and not yet verified against this dataset's actual multi-hot matrices. `SubTabConfig.svd_fields` (`keywords`, `production_companies`) are the two fields large enough to need SVD compression at all -- `genres`/`production_countries`/`spoken_languages` stay multi-hot as-is, small vocabularies. Built via `build_classical_baseline()` (Step 8's classical baseline), sliced to just each field's own multi-hot columns, on `ACTIVE_CANDIDATE`'s actual `train_df`.

In [ ]:
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

from src.features.baselines import build_classical_baseline

classical_baseline = build_classical_baseline(
    train_df,
    train_split,
    fitted_encoders.column_groups.numeric_cols,
    fitted_encoders.standardizer,
    fitted_encoders.language_lookup,
    fitted_encoders.list_poolers,
)

for field in subtab_config.svd_fields:
    field_cols = [c for c in classical_baseline.columns if c.startswith(f"{field}_")]
    # Genuinely sparse -- each row sets only a handful of the ~2000
    # possible items -- so csr_matrix, not a dense array.
    field_matrix = csr_matrix(classical_baseline[field_cols].to_numpy())

    n_components = min(subtab_config.svd_n_components_ceiling, field_matrix.shape[1] - 1)
    # algorithm="arpack", not the default "randomized" -- randomized
    # segfaults on this sparse matrix once n_components reaches ~150 in
    # this environment (isolated separately, not specific to this data);
    # arpack is exact for this problem size and does not hit that crash.
    svd = TruncatedSVD(n_components=n_components, random_state=0, algorithm="arpack")
    svd.fit(field_matrix)
    cumulative_variance = np.cumsum(svd.explained_variance_ratio_)

    fig, ax = plt.subplots()
    ax.plot(range(1, len(cumulative_variance) + 1), cumulative_variance)
    ax.axhline(
        subtab_config.svd_variance_threshold, color="gray", linestyle="--",
        label=f"{subtab_config.svd_variance_threshold:.0%} threshold",
    )
    ax.set_xlabel("number of components (k)")
    ax.set_ylabel("cumulative explained variance")
    ax.set_title(f"{field}: TruncatedSVD cumulative explained variance")
    ax.legend()
    plt.show()

    reached = np.where(cumulative_variance >= subtab_config.svd_variance_threshold)[0]
    if len(reached) > 0:
        k = int(reached[0]) + 1
        print(
            f"{field}: reaches {subtab_config.svd_variance_threshold:.0%} variance "
            f"at k={k} (actual: {cumulative_variance[k-1]:.4f})"
        )
    else:
        print(
            f"{field}: {subtab_config.svd_variance_threshold:.0%} NOT reached by "
            f"k={n_components}; variance at k={n_components} is {cumulative_variance[-1]:.4f}"
        )
        # Curve-shape judgment: compare the variance gained over the last
        # 10% of components against the first 10% -- a concrete "would
        # raising the ceiling help" signal, not just eyeballing the plot.
        n = len(cumulative_variance)
        tail = max(1, n // 10)
        first_gain = cumulative_variance[tail - 1]
        last_gain = cumulative_variance[-1] - cumulative_variance[-1 - tail]
        ratio = last_gain / first_gain
        shape = "still climbing steeply" if ratio > 0.5 else "flattening out"
        print(
            f"  gain from first {tail} components: {first_gain:.4f}, "
            f"gain from last {tail} components: {last_gain:.4f} "
            f"(ratio {ratio:.2f}) -> curve is {shape}"
        )

### Fit list-field SVD reducers (Prompt 3.2)

Refits `SubtabListReducer` for `keywords`/`production_companies` at the exact k Prompt 3.1's diagnostic settled on (`keywords=206`, `production_companies=92`) -- a fresh `TruncatedSVD(n_components=k)` fit, not a slice of the k=300 exploratory fit above, so the variance captured here is the same deterministic `arpack` fit stopped at the k actually used going forward.

In [ ]:
from src.models.subtab import SubtabListReducer

svd_k = {"keywords": 206, "production_companies": 92}

list_reducers = {}
for field, k in svd_k.items():
    reducer = SubtabListReducer().fit(train_df, field, fitted_encoders, k)
    list_reducers[field] = reducer

    variance_captured = reducer.svd.explained_variance_ratio_.sum()
    print(f"{field}: k={k}, variance captured = {variance_captured:.4f}")

    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        transformed = reducer.transform(split_df, fitted_encoders)
        print(f"  {split_name}: shape {transformed.shape}")

### Build flat vectors (Prompt 3.3)

`build_flat_vector()` concatenates every field into the fixed-order reconstruction target for SubTab's autoencoder, using the reducers fit above. `flat_vector_segment_bounds()` returns each segment's column range in that array -- printed here so the layout Prompt 3.6b (`per_segment_reconstruction_mse`) and Prompt 3.7 (`train_subtab`) both rely on is visible before either uses it.

In [ ]:
from src.models.subtab import build_flat_vector, flat_vector_segment_bounds

flat_vectors = {}
for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    flat_vectors[split_name] = build_flat_vector(split_df, fitted_encoders, list_reducers)
    print(f"{split_name}: shape {flat_vectors[split_name].shape}")

segment_bounds = flat_vector_segment_bounds(fitted_encoders, list_reducers)
print("\nsegment bounds:")
for name, (start, end) in segment_bounds.items():
    print(f"  {name}: [{start}, {end})  width={end - start}")

### Column subsets (Prompt 3.4)

`make_column_subsets()` splits the flat vector's columns into `SubTabConfig.n_subsets` overlapping, contiguous subsets -- built once here (width is fixed once `train_df`'s flat vector exists), matching `train_subtab()`'s own once-up-front construction.

In [ ]:
from src.models.subtab import make_column_subsets

full_width = flat_vectors["train"].shape[1]
subsets = make_column_subsets(full_width, subtab_config.n_subsets, subtab_config.overlap)

print(f"n_subsets={subtab_config.n_subsets}, overlap={subtab_config.overlap}, full_width={full_width}\n")
for i, s in enumerate(subsets):
    print(f"subset {i}: size={len(s)}, range=[{s[0]}, {s[-1]}]")
    if i > 0:
        overlap_with_prev = len(set(s.tolist()) & set(subsets[i - 1].tolist()))
        print(f"  overlap with subset {i - 1}: {overlap_with_prev} columns")

### Instantiate SubTabAutoencoder

One shared encoder/decoder pair, sized from `subtab_config.encoder_dims`/`latent_dim` (Uçar et al.'s Income/Obesity architecture) and this dataset's actual `subset_width`/`full_width`.

In [ ]:
from src.models.subtab import SubTabAutoencoder

subset_width = len(subsets[0])
subtab_model = SubTabAutoencoder(subset_width=subset_width, full_width=full_width, config=subtab_config)

n_params = sum(p.numel() for p in subtab_model.parameters())
print(f"SubTabAutoencoder: subset_width={subset_width}, full_width={full_width}")
print(f"parameter count: {n_params:,}")

### Train SubTab

`train_subtab()` trains against `reconstruction_loss` (L_r only, no L_c/L_d -- D28) with early stopping on val L_r, restoring the best epoch's weights -- same discipline as FT-Transformer's own trainer above. Also reports `per_segment_reconstruction_mse`'s two groups (`keywords_and_production_companies` vs `rest`) at that same restored epoch, not the final epoch trained.

In [ ]:
from src.models.train_subtab import train_subtab

subtab_model, subtab_train_losses, subtab_val_losses, subtab_val_segment_mse = train_subtab(
    subtab_model, train_df, val_df, fitted_encoders, list_reducers, subtab_config, shared_training_config
)

print(
    f"Epochs run: {len(subtab_train_losses)} "
    f"(max_epochs={shared_training_config.max_epochs}, patience={shared_training_config.patience})"
)
print(f"Final train L_r: {subtab_train_losses[-1]:.4f}")
print(f"Val L_r (restored/best epoch): {min(subtab_val_losses):.4f}")

best_epoch = int(np.argmin(subtab_val_losses)) + 1
restored_seg = subtab_val_segment_mse[best_epoch - 1]
print(f"Restored epoch {best_epoch} per-segment MSE:")
print(f"  keywords_and_production_companies: {restored_seg['keywords_and_production_companies']:.4f}")
print(f"  rest: {restored_seg['rest']:.4f}")

### Save SubTab embeddings

Encodes all `SubTabConfig.n_subsets` subsets per row via `subtab_model.encoder` (skipping the decoder -- only the latent is needed here) and mean-aggregates them into one `(N, latent_dim)` vector per row. This is Eq. 3.11's own inference-time rule (Uçar et al.'s §2.3 "Test time": "we feed the subsets of test set to the encoder, and get the aggregate of the representations of all available subsets" -- mean aggregation, `SubTabConfig.aggregation="mean"`), not a new choice made here. Each split's embedding matrix is saved to `ARTIFACTS_DIR / "models" / "subtab" / f"{split_name}.parquet"`, mirroring FT-Transformer's own save cell above and `04_feature_encoding.ipynb`'s `RAW_MODEL_DIR`/`CLASSICAL_MODEL_DIR` pattern.

In [ ]:
import torch

SUBTAB_MODEL_DIR = ARTIFACTS_DIR / "models" / "subtab"
SUBTAB_MODEL_DIR.mkdir(parents=True, exist_ok=True)

subset_tensors = [torch.from_numpy(s).long() for s in subsets]

subtab_model.eval()
for split_name, df in FINAL_SPLITS.items():
    flat_t = torch.from_numpy(flat_vectors[split_name]).float()
    embedding_chunks = []
    with torch.no_grad():
        for start in range(0, len(df), subtab_config.batch_size):
            end = min(start + subtab_config.batch_size, len(df))
            chunk = flat_t[start:end]
            subset_latents = [subtab_model.encoder(chunk[:, s]) for s in subset_tensors]
            # Eq. 3.11's own test-time rule -- mean aggregation across
            # all K subsets' latents (Ucar et al. SS2.3 "Test time"),
            # not a new choice made here.
            mean_latent = torch.stack(subset_latents, dim=0).mean(dim=0)
            embedding_chunks.append(mean_latent.numpy())
    embeddings = np.concatenate(embedding_chunks, axis=0)

    embeddings_df = pd.DataFrame(
        embeddings, columns=[f"dim_{i}" for i in range(embeddings.shape[1])]
    )
    path = SUBTAB_MODEL_DIR / f"{split_name}.parquet"
    embeddings_df.to_parquet(path, index=False)
    print(f"Saved: {path} ({embeddings_df.shape})")

## 5. SCARF

### Corruption sanity check (Prompt 4.1)

Corrupts a small sample of `train_df` at `scarf_config.corruption_rate` (0.6) via `corrupt_dataframe()`, then prints the actual fraction of rows that differ per eligible raw column -- each is an independent Bernoulli(`p`) draw, so the observed fraction should land *close to* 0.6, not exactly 0.6. Also confirms `id`/`overview`/`original_title` come through byte-for-byte unchanged, since none of the three is a `corruption_eligible_cols` key.

In [ ]:
from src.models.scarf import corrupt_dataframe

CORRUPTION_SAMPLE_N = 300
corruption_sample_df = train_df.iloc[:CORRUPTION_SAMPLE_N].reset_index(drop=True)
corruption_rng = np.random.default_rng(0)
corrupted_sample_df = corrupt_dataframe(
    corruption_sample_df, fitted_encoders, p=scarf_config.corruption_rate, rng=corruption_rng
)

# corruption_eligible_cols' group keys, expanded to raw df columns the
# same way corrupt_dataframe() itself does internally.
eligible_raw_cols = []
for key in fitted_encoders.corruption_eligible_cols:
    if key == "numeric":
        eligible_raw_cols.extend(fitted_encoders.column_groups.numeric_cols)
        eligible_raw_cols.extend(fitted_encoders.column_groups.bypass_cols)
    elif key == "original_language":
        eligible_raw_cols.append(fitted_encoders.language_lookup.col)
    else:
        eligible_raw_cols.append(fitted_encoders.list_poolers[key].col)


def _values_differ(a, b) -> bool:
    if pd.api.types.is_list_like(a):
        return list(a) != list(b)
    return a != b


print(f"corruption rate p = {scarf_config.corruption_rate}, sample size = {CORRUPTION_SAMPLE_N}\n")
for col in eligible_raw_cols:
    n_diff = sum(
        _values_differ(o, c)
        for o, c in zip(corruption_sample_df[col], corrupted_sample_df[col])
    )
    frac = n_diff / CORRUPTION_SAMPLE_N
    print(f"  {col}: {n_diff}/{CORRUPTION_SAMPLE_N} differ (observed fraction {frac:.3f})")

print()
for col in ["id", "overview", "original_title"]:
    unchanged = (corruption_sample_df[col] == corrupted_sample_df[col]).all()
    print(f"{col} byte-for-byte unchanged: {unchanged}")
    assert unchanged, f"{col} should never be corrupted"

### Instantiate SCARF's tables/encoder/head

`ScarfEmbeddingTables` (Prompt 4.2), `ScarfEncoder` (`f`, Prompt 4.5), and `ScarfProjectionHead` (`g`, Prompt 4.6) -- all three are jointly trained by `train_scarf()` below, so their combined parameter count is printed together.

In [ ]:
from src.models.scarf import (
    ScarfEmbeddingTables,
    ScarfEncoder,
    ScarfProjectionHead,
    build_scarf_flat_vector,
    DEFAULT_E_DIM,
)

scarf_tables = ScarfEmbeddingTables(fitted_encoders, e_dim=DEFAULT_E_DIM)

# build_scarf_flat_vector's width is a fixed function of fitted_encoders/
# scarf_tables, not of which/how-many rows are passed -- a 2-row slice is
# enough to read off ScarfEncoder's required input_dim.
_sample_transformed = fitted_encoders.transform(train_df.iloc[:2])
scarf_flat_width = build_scarf_flat_vector(_sample_transformed, scarf_tables, fitted_encoders).shape[1]

scarf_encoder = ScarfEncoder(input_dim=scarf_flat_width, config=scarf_config)
scarf_encoder_output_dim = scarf_encoder.net[-1].out_features
scarf_head = ScarfProjectionHead(input_dim=scarf_encoder_output_dim, config=scarf_config)

tables_params = sum(p.numel() for p in scarf_tables.parameters())
encoder_params = sum(p.numel() for p in scarf_encoder.parameters())
head_params = sum(p.numel() for p in scarf_head.parameters())

print(f"scarf_flat_width: {scarf_flat_width}")
print(f"scarf_encoder_output_dim: {scarf_encoder_output_dim}")
print(f"ScarfEmbeddingTables params: {tables_params:,}")
print(f"ScarfEncoder params: {encoder_params:,}")
print(f"ScarfProjectionHead params: {head_params:,}")
print(f"total parameters: {tables_params + encoder_params + head_params:,}")

### Train SCARF

`train_scarf()` jointly trains `scarf_tables`/`scarf_encoder`/`scarf_head` against `nt_xent_loss` (Eq. 3.14-3.15), with early stopping on validation NT-Xent using `shared_training_config.patience` -- NOT Bahri et al.'s own reported patience of 3 (see `train_scarf.py`'s module docstring, D12).

In [ ]:
from src.models.train_scarf import train_scarf

scarf_tables, scarf_encoder, scarf_head, scarf_train_losses, scarf_val_losses = train_scarf(
    scarf_tables,
    scarf_encoder,
    scarf_head,
    train_df,
    val_df,
    fitted_encoders,
    scarf_config,
    shared_training_config,
)

print(
    f"Epochs run: {len(scarf_train_losses)} "
    f"(max_epochs={shared_training_config.max_epochs}, patience={shared_training_config.patience})"
)
print(f"Final train NT-Xent: {scarf_train_losses[-1]:.4f}")
print(f"Val NT-Xent (restored/best epoch): {min(scarf_val_losses):.4f}")

### Save SCARF embeddings

Chapter 4's representation is `f`'s output only (Eq. 3.13) -- `g` (`scarf_head`) is discarded here, matching Bahri et al. (2022)'s own "after pre-training, g is discarded" rule (`ScarfProjectionHead`'s own docstring). Runs the UNCORRUPTED flat vector through `scarf_tables` + `scarf_encoder` only -- corruption is a training-time augmentation (Prompt 4.1), not something applied at inference. Each split's `(N, scarf_encoder_output_dim)` embedding matrix is saved to `ARTIFACTS_DIR / "models" / "scarf" / f"{split_name}.parquet"`, mirroring FT-Transformer's/SubTab's own save cells above.

In [ ]:
import torch

SCARF_MODEL_DIR = ARTIFACTS_DIR / "models" / "scarf"
SCARF_MODEL_DIR.mkdir(parents=True, exist_ok=True)

scarf_tables.eval()
scarf_encoder.eval()
for split_name, df in FINAL_SPLITS.items():
    embedding_chunks = []
    with torch.no_grad():
        for start in range(0, len(df), scarf_config.batch_size):
            end = min(start + scarf_config.batch_size, len(df))
            chunk_df = df.iloc[start:end]
            transformed = fitted_encoders.transform(chunk_df)
            flat_vec = build_scarf_flat_vector(transformed, scarf_tables, fitted_encoders)
            embedding_chunks.append(scarf_encoder(flat_vec).numpy())
    embeddings = np.concatenate(embedding_chunks, axis=0)

    embeddings_df = pd.DataFrame(
        embeddings, columns=[f"dim_{i}" for i in range(embeddings.shape[1])]
    )
    path = SCARF_MODEL_DIR / f"{split_name}.parquet"
    embeddings_df.to_parquet(path, index=False)
    print(f"Saved: {path} ({embeddings_df.shape})")